# Fire / Smoke / Normal detector in Colab (model CA)

**Model CA:** resize to 256 (area) → measure noise σ (Immerkær). If σ > 2.56: median 3×3 + CLAHE on L (clip 2.0, tile 8). Otherwise the photo is unchanged. Then MobileViT-XS trained with Gaussian noise.

**How to use:** run the cells from top to bottom. A GPU is **not** needed.

You need two things:
1. **The code.** It is cloned from your GitHub repo `dip_fire_smoke`. If that fails, you upload a zip instead.
2. **The weights** `firesmoke_CA.pt` (about 8 MB). Download them from your v1.0 release, or upload them from your laptop.

## 1. Settings (change only these lines)

In [ ]:
GITHUB_USER = "YOUR_GITHUB_USERNAME"   # <-- type your GitHub username
REPO_NAME   = "dip_fire_smoke"
RELEASE_TAG = "v1.0"                   # tag of the release that has firesmoke_CA.pt
WEIGHTS     = "firesmoke_CA.pt"

## 2. Get the code
First it tries to clone from GitHub. If that fails (private repo or a wrong username), it asks you to upload a zip of your project folder (the folder that has `pyproject.toml` and `firesmoke/`).

In [ ]:
import os, subprocess, zipfile, glob, sys
from google.colab import files

REPO_DIR = f"/content/{REPO_NAME}"
if not os.path.isdir(REPO_DIR):
    ok = subprocess.run(["git", "clone", "-q", f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git", REPO_DIR]).returncode == 0
    if not ok:
        print("Clone failed. Upload a zip of your project folder instead.")
        up = files.upload()
        zipfile.ZipFile(next(iter(up))).extractall("/content/code_upload")
        REPO_DIR = os.path.dirname(glob.glob("/content/code_upload/**/pyproject.toml", recursive=True)[0])

%cd {REPO_DIR}
print("Code folder:", REPO_DIR)

## 3. Install the package
Colab already has torch, so this only adds `timm` and installs `firesmoke`.

In [ ]:
!pip install -q -e .
sys.path.insert(0, REPO_DIR)   # makes the import work without restarting the runtime
!firesmoke --help

## 4. Get the weights
First it tries your GitHub release. If that fails, it asks you to upload `firesmoke_CA.pt` from `E:\\dip_fire_smoke\\release`.

In [ ]:
url = f"https://github.com/{GITHUB_USER}/{REPO_NAME}/releases/download/{RELEASE_TAG}/{WEIGHTS}"
if not os.path.isfile(WEIGHTS):
    ok = subprocess.run(["wget", "-q", "-O", WEIGHTS, url]).returncode == 0 and os.path.getsize(WEIGHTS) > 1_000_000
    if not ok:
        if os.path.exists(WEIGHTS): os.remove(WEIGHTS)
        print("Download failed. Upload firesmoke_CA.pt now.")
        files.upload()
print(WEIGHTS, round(os.path.getsize(WEIGHTS) / 1e6, 1), "MB")

## 5. Upload your photos (a fresh start every run)
This cell **deletes the photos from the last run**, then asks for new ones. Each run also gets its own output folder named with the date and time, so old results are never overwritten.

In [ ]:
import shutil, datetime
shutil.rmtree("photos", ignore_errors=True)        # remove last run's photos
os.makedirs("photos")
RUN_ID  = datetime.datetime.now().strftime("run_%Y%m%d_%H%M%S")
OUT_DIR = f"outputs/{RUN_ID}"                        # new folder for this run

%cd photos
files.upload()
%cd ..
photos = sorted(glob.glob("photos/*"))
print("Run:", RUN_ID)
print(len(photos), "photos:", [os.path.basename(p) for p in photos])

## 6. Run the command line tool on the whole folder
This is the same command as on your laptop. `--steps` saves every intermediate image.

In [ ]:
!firesmoke predict photos --weights {WEIGHTS} --steps --out {OUT_DIR}

## 7. Use it from Python: a results table
One row per photo: label, confidence, measured noise σ, and whether the filter was used.

In [ ]:
import pandas as pd
try:
    from firesmoke import FireSmokeDetector
except ImportError:
    from firesmoke.detector import FireSmokeDetector

det = FireSmokeDetector(WEIGHTS)
results = [det.predict(p) for p in photos]

table = pd.DataFrame({
    "photo":      [os.path.basename(p) for p in photos],
    "label":      [r.label for r in results],
    "confidence": [round(float(r.confidence), 3) for r in results],
    "sigma":      [round(float(r.sigma), 2) for r in results],
    "decision":   [r.decision for r in results],
    "filter":     [r.filter for r in results],
})
os.makedirs(OUT_DIR, exist_ok=True)
table.to_csv(f"{OUT_DIR}/results_table.csv", index=False)
table

## 8. See every step for one photo
Change `i` to look at another photo.

In [ ]:
i = 0
r = results[i]
print(r.explain())
r.show_steps()

## 9. Download this run's outputs as one zip
The zip has the run name, for example `run_20261112_153000.zip`.

In [ ]:
!cd outputs && zip -qr /content/{RUN_ID}.zip {RUN_ID}
files.download(f"/content/{RUN_ID}.zip")